In [1]:
%matplotlib inline
import pystokes
import numpy as np, matplotlib.pyplot as plt

In [2]:
# particle radius, self-propulsion speed, particles fluid viscosity
b, vs, N, eta = 1.0, 0.4, 50000, 0.1

kSpring, kBond = 1, 4                     # kSpring: stiffness of the spring and kBond: is natural length 

# instantiate the libraries
rbm    = pystokes.unbounded.Rbm(radius=b, particles=N, viscosity=eta)
forces = pystokes.forceFields.Forces(particles=N)

In [3]:
r = np.random.random((3*N))


In [4]:
help(forces.beadSpringModel)

Help on method beadSpringModel in module pystokes.forceFields:

beadSpringModel(F, r, bondLength, springModulus, bendModulus) method of pystokes.forceFields.Forces instance
    Force on colloids in a polymers connected by spring

    ...

    Parameters
    ----------
    F: np.array
        An array of forces
        An array of size 3*N,
    r: np.array
        An array of positions
        An array of size 3*N,
    bondLength: float
        The size of natural spring
    springModulus: float
        Stiffness of the spring
    bendModulus: float
        Bending stiffness of the chain



In [5]:
Nf,   bondLength,  springModulus,  bendModulus, twistModulus=1, 1, 1, 1, 0

F = np.zeros((3*N))
print (F)
forces.multipolymers( Nf,   F,  r,  bondLength,  springModulus,  bendModulus, twistModulus)
print (F)

F2 = np.zeros((3*N))
print (F2)
forces.beadSpringModel(F2,  r,  bondLength,  springModulus,  bendModulus)
np.allclose(F, F2)
# print (F2)


[0. 0. 0. ... 0. 0. 0.]
[ 0.52819615 -1.26184     2.62157999 ...  0.34517789 -0.25315867
 -0.07376614]
[0. 0. 0. ... 0. 0. 0.]


True

In [6]:
%%timeit
forces.beadSpringModel(F2,  r,  bondLength,  springModulus,  bendModulus)


2.32 ms ± 124 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [7]:
%%timeit
forces.multipolymers( Nf,   F,  r,  bondLength,  springModulus,  bendModulus, twistModulus)


2.41 ms ± 151 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [8]:
# initialize the positions
r0 = np.zeros(3*N);   r  = np.zeros(3*N)
k=0
for i in range(2):
    for j in range(int(N/2)):
        r0[k]    = -5 + i*10
        r0[k+N] = -5+kBond*j
        k = k+1


def configPlot(r):
    "plot the position of the particles"
    x, y = r[0:N], r[N:2*N]
    plt.plot(x, y, 'o', ms=20)
    plt.xlim(-np.max(x)-20, 20+np.max(x));    plt.ylim(-np.max(y)-20, 20+-np.max(x))

In [9]:
ljeps=.01; ljrmin=4

def simulate(Nt, dt, r):
    F  = np.zeros(3*N)
    v  = np.zeros(3*N)

    for i in range(Nt):
        v, F = v*0, F*0

        forces.lennardJones (F, r, ljeps, ljrmin )
        forces.spring(F, r, kBond, kSpring)
        rbm.mobilityTT(v, r, F)

        r = r + v*dt
        #print (np.max(F), np.max(v), i)

        if i%200==0:
            configPlot(r)
            plt.show()

In [ ]:
Nt, dt = 400, 0.01

# initial condition
r = r0   
configPlot(r)

# simulate and plot
plt.figure()
simulate(Nt, dt, r)